### Generate Samples

In [3]:
import numpy as np
import scipy.stats as stats


# Generate random values from Poisson distribution for East Java
np.random.seed(6)
east_java_ages1 = stats.poisson.rvs(loc=18, mu=30, size=30)
east_java_ages2 = stats.poisson.rvs(loc=18, mu=10, size=20)
east_java_ages = np.concatenate((east_java_ages1, east_java_ages2))

# Generate random values from Poisson distribution for West Java
np.random.seed(12)
west_java_ages1 = stats.poisson.rvs(loc=18, mu=33, size=30)
west_java_ages2 = stats.poisson.rvs(loc=18, mu=13, size=20)
west_java_ages = np.concatenate((west_java_ages1, west_java_ages2))

print( east_java_ages.mean() )
print( west_java_ages.mean() )

40.68
42.8


### Do two-Sample T-Test

In [4]:
stats.ttest_ind(a= east_java_ages,
                b= west_java_ages,
                equal_var=False)    # Assume samples have equal variance false

TtestResult(statistic=-0.9745088195709343, pvalue=0.33221768005728836, df=97.41454338473416)

* pvalue = 0.33 means 33% chance we'd see sample data this far apart if the two groups tested are actually identical.
* Even if we were using a 70% confidence level we would fail to reject (accept) the null hypothesis, since the p-value is greater than the corresponding significance level of 30%. 

# one-way ANOVA

In [ ]:
import scipy.stats as stats

# Sample data for three groups
method_a = [85, 88, 90, 93, 87]
method_b = [78, 82, 84, 80, 79]
method_c = [92, 95, 94, 96, 93]

# Perform one-way ANOVA
f_stat, p_val = stats.f_oneway(method_a, method_b, method_c)
print(f"F-statistic: {f_stat:.2f}, P-value: {p_val:.4f}")

if p_val < 0.05:
    print("The ANOVA results indicate that there is a statistically significant difference among the three methods (p < 0.05).")
else:
    print("The ANOVA results suggest no statistically significant difference among the three methods (p >= 0.05).")

F-statistic: 38.74, P-value: 0.0000
The ANOVA results indicate that there is a statistically significant difference among the three methods (p < 0.05).


# two-way Anova

In [22]:
import pandas as pd
import statsmodels.api as sm
from statsmodels.formula.api import ols
# Create a dataframe for two-way ANOVA using method (A, B, C) and a blocking factor "replicate"
data = {
    'method': ['A'] * 5 + ['B'] * 5 + ['C'] * 5,
    'replicate': list(range(1, 6)) * 3,
    'score': method_a + method_b + method_c
}
df_anova = pd.DataFrame(data)

# Fit the two-way ANOVA model including interaction between method and replicate
model2 = ols('score ~ C(method) + C(replicate)', data=df_anova).fit()
anova_table2 = sm.stats.anova_lm(model2, typ=2)

print(df_anova)
print(anova_table2)

# Interpret the results of the two-way ANOVA (anova_table2)
method_p = anova_table2.loc["C(method)", "PR(>F)"]
replicate_p = anova_table2.loc["C(replicate)", "PR(>F)"]

print("Interpretation of ANOVA results (anova_table2):")
if method_p < 0.05:
    print(f"- The effect of method is statistically significant (p = {method_p:.6f}), suggesting differences in scores across methods A, B, and C.")
else:
    print(f"- The effect of method is not statistically significant (p = {method_p:.6f}).")

if replicate_p < 0.05:
    print(f"- The effect of replicate (blocking factor) is statistically significant (p = {replicate_p:.6f}), indicating variability across replicates.")
else:
    print(f"- The effect of replicate is not statistically significant (p = {replicate_p:.6f}).")

print("- The Residual row shows the unexplained variation in the model.")

   method  replicate  score
0       A          1     85
1       A          2     88
2       A          3     90
3       A          4     93
4       A          5     87
5       B          1     78
6       B          2     82
7       B          3     84
8       B          4     80
9       B          5     79
10      C          1     92
11      C          2     95
12      C          3     94
13      C          4     96
14      C          5     93
                  sum_sq   df          F    PR(>F)
C(method)     454.533333  2.0  82.144578  0.000005
C(replicate)   48.266667  4.0   4.361446  0.036564
Residual       22.133333  8.0        NaN       NaN
Interpretation of ANOVA results (anova_table2):
- The effect of method is statistically significant (p = 0.000005), suggesting differences in scores across methods A, B, and C.
- The effect of replicate (blocking factor) is statistically significant (p = 0.036564), indicating variability across replicates.
- The Residual row shows the unexplained

In [23]:
# Create the dataset
# This dataset represents plant heights based on water and sunlight conditions
# 'water' has two levels: 'daily' and 'weekly'
# 'sun' has three levels: 'low', 'med', and 'high'
# 'height' is the dependent variable representing plant height in cm
df = pd.DataFrame({
    'water': np.repeat(['daily', 'weekly'], 15),
    'sun': np.tile(np.repeat(['low', 'med', 'high'], 5), 2),
    'height': [6, 6, 6, 5, 6, 5, 5, 6, 4, 5, 6, 6, 7, 8, 7,
               3, 4, 4, 4, 5, 4, 4, 4, 4, 4, 5, 6, 6, 7, 8]
})

# Fit the two-way ANOVA model
model = ols('height ~ C(water) + C(sun) + C(water):C(sun)', data=df).fit()
anova_table = sm.stats.anova_lm(model, typ=2)

print(df)

print(anova_table)

print("""\nInterpretation:
- Water factor: p-value = 0.0005 -> significant.
- Sun factor: p-value = 0.0000 -> significant.
- Interaction (Water x Sun): p-value = 0.1207 -> not significant.
""")

     water   sun  height
0    daily   low       6
1    daily   low       6
2    daily   low       6
3    daily   low       5
4    daily   low       6
5    daily   med       5
6    daily   med       5
7    daily   med       6
8    daily   med       4
9    daily   med       5
10   daily  high       6
11   daily  high       6
12   daily  high       7
13   daily  high       8
14   daily  high       7
15  weekly   low       3
16  weekly   low       4
17  weekly   low       4
18  weekly   low       4
19  weekly   low       5
20  weekly   med       4
21  weekly   med       4
22  weekly   med       4
23  weekly   med       4
24  weekly   med       4
25  weekly  high       5
26  weekly  high       6
27  weekly  high       6
28  weekly  high       7
29  weekly  high       8
                    sum_sq    df        F    PR(>F)
C(water)          8.533333   1.0  16.0000  0.000527
C(sun)           24.866667   2.0  23.3125  0.000002
C(water):C(sun)   2.466667   2.0   2.3125  0.120667
Residual         

# Chi-Square Test 

In [18]:
import pandas as pd
from scipy.stats import chi2_contingency

# Create a contingency table
data = [[30, 10, 20],   # Under 18: Vanilla, Choco, Strawberry
        [25, 30, 15],   # 18-35
        [20, 25, 30]]   # 36+

df = pd.DataFrame(data, columns=['Vanilla', 'Chocolate', 'Strawberry'])

# Perform Chi-Square Test of Independence
chi2, p, dof, expected = chi2_contingency(df)
print(f"Chi2 Statistic: {chi2:.2f}, P-value: {p:.4f}")

print("Null hypothesis: There is no association between age groups and ice cream flavors; the variables are independent.")

# Interpretation of the Chi-Square Test results:
if p < 0.05:
    print(f"Chi-Square Test: With a chi2 statistic of {chi2:.2f} and a p-value of {p:.4f}, \nWe reject the null hypothesis. This indicates a statistically significant association between the age groups and the ice cream flavors.")
else:
    print(f"Chi-Square Test: With a chi2 statistic of {chi2:.2f} and a p-value of {p:.4f}, \nWe fail to reject the null hypothesis. This suggests that there is no significant association between the age groups and the ice cream flavors.")

Chi2 Statistic: 16.08, P-value: 0.0029
Null hypothesis: There is no association between age groups and ice cream flavors; the variables are independent.
Chi-Square Test: With a chi2 statistic of 16.08 and a p-value of 0.0029, 
We reject the null hypothesis. This indicates a statistically significant association between the age groups and the ice cream flavors.


<b>Summary:</b><br>
ANOVA compares group means in continuous data,<br>
Chi-Square compares group frequencies in categorical data.
